# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset name and description (accessing as attributes)
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and their `@id`s.

In [ ]:
# List record sets with @id, name, and description
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  name: {rs.get('name', '[no name]')}")
    print(f"  description: {rs.get('description', '[no description]')}")
    # Print fields
    print("  Fields:")
    for field in rs.get('field', []):
        if isinstance(field, dict):
            print(f"    Field @id: {field.get('@id')} | name: {field.get('name', '[no name]')}")
        else:
            print(f"    Field (reference): {field}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect the @id of all record sets
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

# Extract records for each record set, loading into dataframes
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# For demonstration, select the first record set if available
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"Fields in record set '{example_record_set_id}':")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
# Select a record set with numeric data for demonstration
if record_set_ids:
    df = dataframes[example_record_set_id]
"""
Suppose one numeric field is named by its @id, e.g., 'cr:field/coverage_percentage'.
Below, we try to auto-detect a field with numeric data for EDA demonstration. You may update this field based on actual columns.
"""
    numeric_field = None
    # Try to find a numeric column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        # Try to convert columns to numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            except Exception:
                continue
        # Try finding numeric dtype again
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
    if numeric_field:
        print(f"Using numeric field '@id': {numeric_field}")
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a non-numeric field (@id)
        group_field = None
        for col in df.columns:
            if col != numeric_field and pd.api.types.is_object_dtype(df[col]):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
    else:
        print("No numeric field found for analysis. Please check the DataFrame.")
else:
    print('No data available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and 'numeric_field' in locals() and numeric_field:
    # Histogram of the numeric field
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, color='skyblue')
    plt.title(f"Distribution of field: {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group if available
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(12,6))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR² formatted mass spectrometry dataset using Croissant tools. You can extend the analyses by leveraging other fields (referenced by their `@id`s) and more advanced transformations or visual analytics.

**Key findings and next steps:**
- The dataset structure and fields are accessible and interpretable using the Croissant schema.
- Basic filtering and normalization operations can be customized for specific fields of interest, referenced always by their `@id`.
- For more advanced workflows, refer to the full list of fields and record sets in Section 2, and always operate on data by referencing the `@id` attributes.